In [1]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os

os.listdir("./outputs")

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error, r2_score
from scipy.stats import pearsonr
import polars as pl
import numpy as np
from tqdm import tqdm
from datetime import datetime
import os

train_splits = {
    "full" : pl.datetime(2023, 3, 1, 0, 0, 0),
    "last_12m" : pl.datetime(2023, 6, 1, 0, 0, 0),
    "last_9m" : pl.datetime(2023, 9, 1, 0, 0, 0),
    "last_3m" : pl.datetime(2023, 12, 1, 0, 0, 0),
    "last_1m": pl.datetime(2024, 2, 1, 0, 0, 0),
}

PATHS = {
    "TRAIN_PATH" :"./kaggle/kaggle/input/drw-crypto-market-prediction/train.parquet",
    "TEST_PATH" : "./kaggle/kaggle/input/drw-crypto-market-prediction/test.parquet",
    "SUBMISSION_PATH" : "/kaggle/input/drw-crypto-market-prediction/sample_submission.csv",
}

train_data = pl.read_parquet(PATHS["TRAIN_PATH"]).sort("timestamp", descending = False)
print(train_data)

shape: (525_887, 897)
┌─────────┬─────────┬─────────┬──────────┬───┬──────────┬──────────┬──────────┬──────────────┐
│ bid_qty ┆ ask_qty ┆ buy_qty ┆ sell_qty ┆ … ┆ X889     ┆ X890     ┆ label    ┆ timestamp    │
│ ---     ┆ ---     ┆ ---     ┆ ---      ┆   ┆ ---      ┆ ---      ┆ ---      ┆ ---          │
│ f64     ┆ f64     ┆ f64     ┆ f64      ┆   ┆ f64      ┆ f64      ┆ f64      ┆ datetime[ns] │
╞═════════╪═════════╪═════════╪══════════╪═══╪══════════╪══════════╪══════════╪══════════════╡
│ 15.283  ┆ 8.425   ┆ 176.405 ┆ 44.984   ┆ … ┆ 0.159183 ┆ 0.530636 ┆ 0.562539 ┆ 2023-03-01   │
│         ┆         ┆         ┆          ┆   ┆          ┆          ┆          ┆ 00:00:00     │
│ 38.59   ┆ 2.336   ┆ 525.846 ┆ 321.95   ┆ … ┆ 0.158963 ┆ 0.530269 ┆ 0.533686 ┆ 2023-03-01   │
│         ┆         ┆         ┆          ┆   ┆          ┆          ┆          ┆ 00:01:00     │
│ 0.442   ┆ 60.25   ┆ 159.227 ┆ 136.369  ┆ … ┆ 0.158744 ┆ 0.529901 ┆ 0.546505 ┆ 2023-03-01   │
│         ┆         ┆       

# Data

In [2]:
def get_cols_inf(df: pl.DataFrame) -> list[str]:
    """
    Returns a list of column names that contain any positive or negative infinity.
    """
    cols = []
    for col in df.columns:
        # df[col] is a Series; .is_infinite() → Boolean Series; .any() → Python bool
        try:
            if df[col].is_infinite().any():
                cols.append(col)
        except Exception:
            # if the column isn’t numeric, .is_infinite() might error—just skip it
            continue
    return cols

def get_nan_columns(df: pl.DataFrame) -> list[str]:
    """
    Returns a list of column names with any NaN/null values.
    """
    cols = []
    for col in df.columns:
        if df.select(pl.col(col).is_null().any()).item():
            cols.append(col)
    return cols

def get_cols_zerostd(df: pl.DataFrame) -> list[str]:
    """
    Returns a list of column names whose standard deviation is zero
    (or whose std returns None because all values are null).
    Non-numeric columns (e.g. datetime) are skipped.
    """
    cols = []
    for col, dtype in zip(df.columns, df.dtypes):
        # Only attempt std() on numeric dtypes
        if dtype.is_numeric():  
            # df[col] is a Series; .std() returns a Python float or None
            std_val = df[col].std()
            if std_val == 0.0 or std_val is None:
                cols.append(col)
    return cols


def feature_engineering(df: pl.DataFrame) -> pl.DataFrame:
    # Feature engineering
    df = df.with_columns([
        # bidask_ratio = bid_qty / ask_qty
        (pl.col("bid_qty") / pl.col("ask_qty")).alias("bidask_ratio"),

        # buysell_ratio = 0 if volume == 0 else buy_qty / sell_qty
        pl.when(pl.col("volume") == 0)
        .then(0)
        .otherwise(pl.col("buy_qty") / pl.col("sell_qty"))
        .alias("buysell_ratio"),

        # bidask_delta = bid_qty - ask_qty
        (pl.col("bid_qty") - pl.col("ask_qty")).alias("bidask_delta"),

        # buysell_delta = buy_qty - sell_qty
        (pl.col("buy_qty") - pl.col("sell_qty")).alias("buysell_delta"),

        # buysell_size = buy_qty + sell_qty
        (pl.col("buy_qty") + pl.col("sell_qty")).alias("buysell_size"),

        # bidask_size = bid_qty + ask_qty
        (pl.col("bid_qty") + pl.col("ask_qty")).alias("bidask_size"),
    ])
    return df
def preprocess_train(train: pl.DataFrame, columns_to_drop: list[str] = []) -> pl.DataFrame:
    """
    Mirror of the original pandas workflow, but using polars.
    1. Identify columns with infinite, NaN, or zero‐std and drop them.
    2. Drop any user‐specified columns (e.g. label or order‐book columns).
    3. (You can add normalized/scaling steps here if needed.)
    """
    df = train.clone()

    df = feature_engineering(df)
    
    #### Preprocessing
    cols_inf = get_cols_inf(df)
    print("Columns with infinite values:", cols_inf)

    cols_nan = get_nan_columns(df)
    print("Columns with NaN values:", cols_nan)

    cols_zerostd = get_cols_zerostd(df)
    print("Columns with zero standard deviation:", cols_zerostd)
    # Drop columns with infinite, NaN, or zero‐std values
    drop_columns = list(set(cols_inf) | set(cols_nan) | set(cols_zerostd) | set(columns_to_drop))
    if drop_columns:
        df = df.drop(drop_columns)
    # df = df.sort("timestamp", descending=False)
    return df, drop_columns

def preprocess_test(test: pl.DataFrame, columns_to_drop: list[str] = []) -> pl.DataFrame:
    df = test.clone()
    df = feature_engineering(df)
    df = df.drop(columns_to_drop)
    print("Columns dropped from test set:", columns_to_drop)
    return df

# 1 Basic Feature Selection

- Correlation : No significant correlation with features.

In [3]:
X_data, drop_columns  = preprocess_train(train_data, columns_to_drop=["bid_qty", "ask_qty", "buy_qty", "sell_qty"])
print(X_data)

Columns with infinite values: ['X697', 'X698', 'X699', 'X700', 'X701', 'X702', 'X703', 'X704', 'X705', 'X706', 'X707', 'X708', 'X709', 'X710', 'X711', 'X712', 'X713', 'X714', 'X715', 'X716', 'X717']
Columns with NaN values: []
Columns with zero standard deviation: ['X864', 'X867', 'X869', 'X870', 'X871', 'X872']
shape: (525_887, 872)
┌─────────┬──────────┬───────────┬───────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ volume  ┆ X1       ┆ X2        ┆ X3        ┆ … ┆ bidask_del ┆ buysell_de ┆ buysell_s ┆ bidask_si │
│ ---     ┆ ---      ┆ ---       ┆ ---       ┆   ┆ ta         ┆ lta        ┆ ize       ┆ ze        │
│ f64     ┆ f64      ┆ f64       ┆ f64       ┆   ┆ ---        ┆ ---        ┆ ---       ┆ ---       │
│         ┆          ┆           ┆           ┆   ┆ f64        ┆ f64        ┆ f64       ┆ f64       │
╞═════════╪══════════╪═══════════╪═══════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ 221.389 ┆ 0.121263 ┆ -0.41769  ┆ 0.005399  ┆ … ┆ 6.858  

In [ ]:
cols = train_data.drop("timestamp").columns
inf_flags = train_data.select([pl.col(c).is_infinite().any().alias(c) for c in cols]).row(0)
inf_cols = [c for c, flag in zip(cols, inf_flags) if flag]
inf_cols

['X697',
 'X698',
 'X699',
 'X700',
 'X701',
 'X702',
 'X703',
 'X704',
 'X705',
 'X706',
 'X707',
 'X708',
 'X709',
 'X710',
 'X711',
 'X712',
 'X713',
 'X714',
 'X715',
 'X716',
 'X717']

In [ ]:
# for t_name, t_split in train_splits.items():
#     print(f"Processing split: {t_name} at {t_split}")
#     X_filtered = X_data.filter(pl.col("timestamp") >= t_split)
#     correlation = X_filtered.drop(["timestamp"]).corr()
#     feats = [c for c in X_filtered.columns if c not in ("timestamp", "label")]
#     corr_with_label = X_filtered.select([
#         pl.corr(f, "label").alias(f)
#         for f in feats
#     ]).unpivot(
#         on = [],
#         variable_name = "feature",
#         value_name = "correlation"
#     ).sort("correlation", descending=True)
#     print(corr_with_label)

Processing split: full at 2023-03-01 00:00:00.alias("datetime")
shape: (870, 2)
┌─────────┬─────────────┐
│ feature ┆ correlation │
│ ---     ┆ ---         │
│ str     ┆ f64         │
╞═════════╪═════════════╡
│ X21     ┆ 0.069401    │
│ X20     ┆ 0.067667    │
│ X28     ┆ 0.064092    │
│ X863    ┆ 0.064057    │
│ X29     ┆ 0.062339    │
│ …       ┆ …           │
│ X580    ┆ -0.041725   │
│ X95     ┆ -0.042948   │
│ X137    ┆ -0.04429    │
│ X524    ┆ -0.04802    │
│ X531    ┆ -0.056184   │
└─────────┴─────────────┘
Processing split: last_12m at 2023-06-01 00:00:00.alias("datetime")
shape: (870, 2)
┌─────────┬─────────────┐
│ feature ┆ correlation │
│ ---     ┆ ---         │
│ str     ┆ f64         │
╞═════════╪═════════════╡
│ X863    ┆ 0.061713    │
│ X21     ┆ 0.061699    │
│ X20     ┆ 0.058837    │
│ X856    ┆ 0.055154    │
│ X598    ┆ 0.054378    │
│ …       ┆ …           │
│ X95     ┆ -0.038783   │
│ X576    ┆ -0.04029    │
│ X137    ┆ -0.041157   │
│ X504    ┆ -0.041689   │
│ X5

In [ ]:
class FeatureEngineeringPipeline:
    def __init__(
        self,
        data: pl.DataFrame,
        y_col: str = "label",
        drop_columns: list[str] = None,
        config: dict = None
    ):
        """
        config = {
            "model_agnostic": {
                "preprocessing": {…},
                "transformation": {"poly_degree": 2, …},
                "filter": {"method": "SelectPercentile", "percentile": 10},
                "extraction": {"method": "PCA", "n_components": 5}
            },
            "model_based": {
                "embedded": {"alpha": 1.0},
                "wrapper": {"n_features_to_select": 10},
                "perm_importance": {"n_repeats": 5},
                "stability": {"n_bootstrap": 50}
            },
            "aggregation": {"strategy": "rank_sum", "weights": {...}}
        }
        """
        self.data = data
        self.y_col = y_col
        self.drop_columns = drop_columns or []
        self.config = config or {}


    def run_feature_engineering_pipeline(self):
        """
        A comprehensive quantitative feature engineering pipeline.
        """
        pass
    
    def run_model_agnostic_selection(self, lazy_df: pl.DataFrame | pl.LazyFrame):
        """
        Pre-model evaluation:
        1. Data Preprocessing 
            Handle missing, infinite or outlier values to ensure numerical stability (e.g. drop or impute NaNs, clip infinities or extreme quantiles).
        2. Scaling/Normalization
            Standardize variances (z-score) or map to fixed ranges (min-max, log-scaling) so all features live on comparable numerical scales.
        3. Add Features / Categorical Encoding
            Convert categoricals (one-hot, ordinal), date/times (e.g. cyclic encodings), and build simple interactions or aggregated statistics (ratios, deltas).
        4. Feature Transformation / Generation
            - Transformation: apply deterministic mappings column-wise—e.g. PolynomialFeatures, FunctionTransformer, PowerTransformer, QuantileTransformer, or hash-based projections (FeatureHasher).
            - Generation: assemble parallel or heterogeneous pipelines via FeatureUnion or ColumnTransformer to combine, conditionally apply or concatenate multiple transforms.
        5. Unsupervised Extraction
            - Extraction: reduce dimensionality by projecting into latent subspaces (PCA, TruncatedSVD, ICA, NMF) using only input covariances or non-negativity constraints.
        6. Filter methods 
            - Filter selection: remove low-value inputs via univariate criteria—variance thresholds, correlation filters, or statistical tests with error-rate control (e.g. SelectKBest, SelectPercentile, SelectFwe, SelectFdr).
        """
        # 1. Data Preprocessing
        lazy_df, drop_list = self._data_preprocessing_1(lazy_df)
        # 2. Scaling/Normalization

        # 3. Add Features / Categorical Encoding

        # 4. Feature Transformation / Generation

        # 5. Unsupervised Extraction

        # 6. Filter methods

    def run_model_based_selection(self):
        """
        Model evaluation:
        1. Embedded Methods - Lasso, Importance
        2. Wrapper - RFE, sequential
        3. Model agnostic importance - SHAP, Permutation Importance
        4. Stability Selection - 
        """
        pass

    def _data_preprocessing_1(self, lazy_df: pl.DataFrame | pl.LazyFrame) -> tuple[pl.DataFrame, list[str]]:
        cols = lazy_df.drop("timestamp").columns
        inf_flags = lazy_df.select([pl.col(c).is_infinite().any().alias(c) for c in cols]).row(0)
        inf_cols = [c for c, flag in zip(cols, inf_flags) if flag]

        print("1.1 ")
        nan_flags = lazy_df.select([pl.col(c).is_null().any().alias(c) for c in cols]).collect().row(0)
        nan_cols = [c for c, flag in zip(cols, nan_flags) if flag]

        numeric_cols = [c for c, dt in zip(lazy_df.columns, lazy_df.dtypes) if dt.is_numeric()]
        std_flags = (
            lazy_df
            .select([pl.col(c).std().alias(c) for c in numeric_cols])
            .collect()
            .row(0)
        )
        zerostd_cols = [c for c, std in zip(numeric_cols, std_flags) if std == 0 or std is None]

        drop_list = list(set(inf_cols + nan_cols + zerostd_cols))
        return lazy_df.drop(drop_list), drop_list
    
    def __get_config(self, key: str, default=None):
        """
        Retrieve a configuration value, with optional default.
        """
        return self.config.get(key, default)
    